# 🚀 95GB Colab 远程算力管理基座 (AI Live Remote Agent)

**只需在 Google Colab 运行一次（约 20 秒启动，无需提前等待下载任何大模型！）**

运行后，复制下方打印出的 **远程管理地址** 和 **ADMIN_TOKEN** 发给助手，之后即可像远程 Linux GPU 服务器一样由助手自动帮你：
- 在云端安装并运行 **Qwen3-TTS-12Hz-1.7B-VoiceDesign**（凭空捏出 8 位超幼态软萌宝宝音色）
- 运行 **IndexTTS-2 / GPT-SoVITS**（情感克隆 + 导出本地 CPU 可用的 ONNX 权重）
- 运行 **TRELLIS.2 / BiRefNet**（生成 3D `.glb` 家具道具与角色模型并回传到本地永久免云端使用）

In [ ]:
# @title 1️⃣ 20秒极速启动远程管理基座（点击左侧运行或按 Ctrl+F9）
!pip install -q fastapi uvicorn python-multipart
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

import os, re, time, json, secrets, subprocess, threading, asyncio
from pathlib import Path
import torch
from fastapi import FastAPI, Header, HTTPException, UploadFile, File, Form
from fastapi.responses import FileResponse, JSONResponse
from pydantic import BaseModel
import uvicorn

ADMIN_TOKEN = secrets.token_urlsafe(24)
WORK_ROOT = Path('/content/ai_live_cloud')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
BOOT_TS = time.time()

app = FastAPI(title='Personal Colab GPU Remote Agent')

def check_auth(authorization: str = ''):
    tok = authorization.replace('Bearer ', '').strip() if authorization else ''
    if not tok or not secrets.compare_digest(tok, ADMIN_TOKEN):
        raise HTTPException(status_code=401, detail='Unauthorized')

def gpu_info():
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        return {
            'gpu_name': p.name,
            'vram_used_gb': round(torch.cuda.memory_allocated(0) / (1024**3), 2),
            'vram_total_gb': round(p.total_memory / (1024**3), 2),
            'cuda_version': torch.version.cuda,
        }
    return {'gpu_name': 'CPU', 'vram_used_gb': 0.0, 'vram_total_gb': 0.0}

@app.get('/health')
async def health():
    return {
        'ok': True,
        'mode': 'remote_agent_v1',
        'uptime_sec': int(time.time() - BOOT_TS),
        'work_root': str(WORK_ROOT),
        'gpu': gpu_info(),
    }

class ExecReq(BaseModel):
    cmd: str
    cwd: str = '/content/ai_live_cloud'
    timeout: int = 300
    background: bool = False
    log_file: str = ''

@app.post('/admin/exec')
async def admin_exec(req: ExecReq, authorization: str = Header(default='')):
    check_auth(authorization)
    cwd = Path(req.cwd)
    cwd.mkdir(parents=True, exist_ok=True)
    if req.background:
        lf = req.log_file or str(WORK_ROOT / f'bg_{int(time.time())}.log')
        f = open(lf, 'w', encoding='utf-8')
        proc = subprocess.Popen(req.cmd, shell=True, cwd=str(cwd), stdout=f, stderr=subprocess.STDOUT)
        return {'ok': True, 'background': True, 'pid': proc.pid, 'log_file': lf}
    def _run():
        return subprocess.run(
            req.cmd, shell=True, cwd=str(cwd),
            capture_output=True, text=True, timeout=min(1800, max(5, req.timeout))
        )
    try:
        cp = await asyncio.to_thread(_run)
        return {
            'ok': cp.returncode == 0,
            'returncode': cp.returncode,
            'stdout': cp.stdout[-12000:],
            'stderr': cp.stderr[-6000:],
        }
    except subprocess.TimeoutExpired as te:
        return JSONResponse({'ok': False, 'error': 'timeout', 'stdout': str(te.stdout or '')[-4000:]}, status_code=504)

@app.post('/admin/upload')
async def admin_upload(path: str = Form(...), file: UploadFile = File(...), authorization: str = Header(default='')):
    check_auth(authorization)
    dst = Path(path)
    if not dst.is_absolute():
        dst = WORK_ROOT / dst
    dst.parent.mkdir(parents=True, exist_ok=True)
    data = await file.read()
    dst.write_bytes(data)
    return {'ok': True, 'saved': str(dst), 'bytes': len(data)}

@app.get('/admin/download')
async def admin_download(path: str, authorization: str = Header(default='')):
    check_auth(authorization)
    src = Path(path)
    if not src.is_absolute():
        src = WORK_ROOT / src
    if not src.exists() or not src.is_file():
        raise HTTPException(status_code=404, detail='File not found')
    return FileResponse(str(src), filename=src.name)

@app.get('/admin/ls')
async def admin_ls(path: str = '/content/ai_live_cloud', authorization: str = Header(default='')):
    check_auth(authorization)
    p = Path(path)
    if not p.exists():
        return {'ok': False, 'items': []}
    items = [{'name': x.name, 'is_dir': x.is_dir(), 'size': x.stat().st_size if x.is_file() else 0} for x in sorted(p.iterdir())]
    return {'ok': True, 'path': str(p), 'items': items}

# 启动管理服务 (7861) + 自动拉起公网隧道
threading.Thread(target=lambda: uvicorn.run(app, host='127.0.0.1', port=7861, log_level='warning'), daemon=True).start()
time.sleep(2)
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7861'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
public_url = ''
for _ in range(80):
    line = tunnel.stdout.readline()
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        break

print('=' * 72)
print('✅ 95GB Colab 远程算力管理基座已就绪！')
print('🔗 请把下面两行直接复制发给助手，此后云端装模型/捏声音/做3D全自动完成：')
print('   远程管理地址:', public_url)
print('   ADMIN_TOKEN:', ADMIN_TOKEN)
print('=' * 72)
while True:
    time.sleep(60)
